# **API de lluvias**

##### **Group 10** - Temporada: 1 de enero – 31 de mayo de 2023
- Nicolas Silva
- Marco Virú
- Estefanny Mejía
- Luz Supo

Usamos la API de Open-Meteo para medir cuánto llovió en la capital de cada departamento

Primero cargamos las librerías

In [26]:
import io
import time
import requests
import pandas as pd

**1. Tabla de departamentos y capitales (Wikipedia)**

In [27]:
url = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"
respuesta = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
respuesta.raise_for_status()

tablas = pd.read_html(io.StringIO(respuesta.text))
len(tablas)

6

**2. Verificación de 25 departamentos y capitales**

In [28]:
for i, tabla in enumerate(tablas):
    print(i, tabla.shape, list(tabla.columns)[:6])

0 (1, 1) ['Departamentos del Perú (con las provincias de régimen especial)']
1 (24, 11) ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
2 (8, 5) ['Departamento', 'Capital', 'Creación', 'Desaparición', 'Causa']
3 (2, 11) ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
4 (6, 4) ['Provincia', 'Capital', 'Creación', 'Desaparición']
5 (1, 2) [0, 1]


La tabla 1 tiene las columnas `Departamento` y `Capital`.

In [29]:
capitales = tablas[1][["Departamento", "Capital"]].copy()
capitales.columns = ["departamento", "capital"]
capitales

,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


In [30]:
print("Filas:", len(capitales))

Filas: 24


La tabla solo tiene **24 filas** (falta el **Callao**). En Wikipedia aparece en otra tabla, la de provincias de régimen especial (`tablas[3]`).

In [31]:
tablas[3][["Provincia", "Capital"]]

,Provincia,Capital
0,Callao[nota 6],Callao
1,Lima[nota 7],Lima


In [32]:
callao = pd.DataFrame({"departamento": ["Callao"], "capital": ["Callao"]})
capitales = pd.concat([capitales, callao], ignore_index=True)

# Lima aparece como "Huacho (de facto)": usamos Lima
capitales["capital"] = capitales["capital"].replace("Huacho (de facto)", "Lima")

print("Filas:", len(capitales))
capitales

Filas: 25


,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


- **Callao:** lo agregamos a mano con los datos de la tabla de provincias de régimen especial de la misma página de Wikipedia (capital: Callao).
- **Lima:** la capital aparecía como *"Huacho (de facto)"*, que la API no encontraría. Decidimos usar **Lima**, porque es la ciudad principal del departamento y donde vive la mayoría de su población. Huacho es la sede del Gobierno Regional de Lima Provincias.

**3. Geocodificación de las capitales**

Probamos con una sola capital para ver qué devuelve la API.

In [33]:
url_geo = "https://geocoding-api.open-meteo.com/v1/search"

r = requests.get(url_geo, params={"name": "Chachapoyas", "count": 10, "language": "es"}, timeout=30)
r.raise_for_status()
datos = r.json()
datos.keys()

dict_keys(['results', 'generationtime_ms'])

  Cada resultado aquí es un diccionario. Las variables que nos interesan son:

  | Variable | Qué contiene |
  |---|---|
  | `name` | Nombre del lugar encontrado |
  | `country_code` | Código del país (`PE` = Perú) |
  | `admin1` | Departamento o región donde está el lugar |
  | `latitude` | Latitud |
  | `longitude` | Longitud |

In [34]:
pd.DataFrame(datos["results"])[["name", "country_code", "admin1", "latitude", "longitude"]]

,name,country_code,admin1,latitude,longitude
0,Chachapoyas,PE,Departamento de Amazonas,-6.23169,-77.86903
1,Destilería Chachapoyas,AR,Provincia de Salta,-24.74735,-65.38527
2,Chachapoyas,PE,Ancash,-8.97357,-78.49754
3,Chachapoyas,PE,Departamento de Amazonas,-6.20181,-77.85606


La API devuelve hasta 10 lugares con el mismo nombre, dentro y fuera del Perú. Por ejemplo, al buscar *Chachapoyas* apareció un resultado en Argentina ("Destilería Chachapoyas", `AR`), y para *Arequipa* también aparecen lugares en Cajamarca y Loreto.

Por eso nos quedamos con el primer resultado que tiene `country_code == "PE"` y luego comparamos `admin1` con el departamento.

In [35]:
def geocodificar(capital):
    """Devuelve el primer resultado de la API que está en el Perú."""
    r = requests.get(url_geo, params={"name": capital, "count": 10, "language": "es"}, timeout=30)
    r.raise_for_status()
    for resultado in r.json().get("results", []):
        if resultado["country_code"] == "PE":
            return resultado
    return None


geocodificar("Arequipa") #Probamos con Arequipa

{'id': 3947322,
 'name': 'Arequipa',
 'latitude': -16.39899,
 'longitude': -71.53747,
 'elevation': 2397.0,
 'feature_code': 'PPLA',
 'country_code': 'PE',
 'admin1_id': 3947319,
 'admin2_id': 3947318,
 'admin3_id': 8350798,
 'timezone': 'America/Lima',
 'population': 1008290,
 'country_id': 3932488,
 'country': 'Perú',
 'admin1': 'Departamento de Arequipa',
 'admin2': 'Provincia de Arequipa',
 'admin3': 'Arequipa'}

In [36]:
filas = []
for capital in capitales["capital"]:
    resultado = geocodificar(capital)
    filas.append({
        "capital": capital,
        "admin1": resultado["admin1"],
        "latitud": resultado["latitude"],
        "longitud": resultado["longitude"],
    })
    time.sleep(1)      # pausa entre pedidos

coordenadas = pd.DataFrame(filas)
capitales = capitales.merge(coordenadas, on="capital", how="left")
capitales

,departamento,capital,admin1,latitud,longitud
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422


**4. Verificación: todas las capitales en el Perú y en el departamento correcto**

`admin1` no viene escrito igual que nuestro departamento (*"Departamento de Cusco"*, *"Ancash"* sin tilde, *"Provincia Constitucional del Callao"*, *"Provincia de Lima"*). Para compararlos usamos la función `normalizar` y quitamos los prefijos.

In [37]:
def normalizar(serie):
    """Mayúsculas, sin tildes y sin espacios sobrantes"""
    return (
        serie.str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.upper()
        .str.strip()
    )


admin1_limpio = (
    capitales["admin1"]
    .str.replace("Departamento de ", "")
    .str.replace("Provincia Constitucional del ", "")
    .str.replace("Provincia de ", "")
)
capitales["coincide"] = normalizar(capitales["departamento"]) == normalizar(admin1_limpio)

print("Filas que no coinciden:", (~capitales["coincide"]).sum())
capitales[["departamento", "capital", "admin1", "latitud", "longitud", "coincide"]]

Filas que no coinciden: 0


,departamento,capital,admin1,latitud,longitud,coincide
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,True
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869,True
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,True
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,True
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,True
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027,True
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701,True
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298,True
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989,True
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422,True


La API devuelve hasta 10 lugares con el mismo nombre (por ejemplo, para *Arequipa* también aparecen lugares en Cajamarca y Loreto). Nos quedamos con el primero que tiene `country_code == "PE"` y luego comparamos `admin1` con el departamento. Revisamos la tabla fila por fila: las 25 capitales quedaron en el Perú y en su departamento.

**5. Lluvia diaria de la temporada (API histórica)**


In [38]:
#Probamos primero con Chachapoyas (-6.23, -77.87)
url_lluvia = "https://archive-api.open-meteo.com/v1/archive"

parametros = {
    "latitude": -6.23,
    "longitude": -77.87,
    "start_date": "2023-01-01",
    "end_date": "2023-05-31",
    "daily": "precipitation_sum",
    "timezone": "America/Lima",
}

r = requests.get(url_lluvia, params=parametros, timeout=30)
r.raise_for_status()
datos = r.json()
datos.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])

In [39]:
prueba = pd.DataFrame(datos["daily"])
print("Días:", len(prueba))
prueba.head()

Días: 151


,time,precipitation_sum
0,2023-01-01,9.8
1,2023-01-02,0.9
2,2023-01-03,0.1
3,2023-01-04,0.2
4,2023-01-05,0.0


Ahora lo mismo para las 25 capitales. Juntamos todos los días en una sola tabla, con una columna que dice a qué departamento pertenece cada día

In [40]:
def pedir_lluvia(latitud, longitud):
    """Trae la lluvia diaria de la temporada 2023 como DataFrame"""
    parametros = {
        "latitude": latitud,
        "longitude": longitud,
        "start_date": "2023-01-01",
        "end_date": "2023-05-31",
        "daily": "precipitation_sum",
        "timezone": "America/Lima",
    }
    r = requests.get(url_lluvia, params=parametros, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()["daily"])


tablas_lluvia = []
for i, fila in capitales.iterrows():
    diaria = pedir_lluvia(fila["latitud"], fila["longitud"])
    diaria["departamento"] = fila["departamento"]
    tablas_lluvia.append(diaria)
    time.sleep(1)      # pausa entre pedidos

lluvia_diaria = pd.concat(tablas_lluvia, ignore_index=True)
lluvia_diaria.columns = ["fecha", "lluvia_mm", "departamento"]
print("Filas:", len(lluvia_diaria), "(25 departamentos x 151 días)")
lluvia_diaria.head()

Filas: 3775 (25 departamentos x 151 días)


,fecha,lluvia_mm,departamento
0,2023-01-01,9.8,Amazonas
1,2023-01-02,0.9,Amazonas
2,2023-01-03,0.1,Amazonas
3,2023-01-04,0.2,Amazonas
4,2023-01-05,0.0,Amazonas


**6. Lluvia total y días de lluvia fuerte por departamento**

In [41]:
lluvia_diaria["lluvia_fuerte"] = lluvia_diaria["lluvia_mm"] >= 20

resumen = lluvia_diaria.groupby("departamento").agg(
    lluvia_total_mm=("lluvia_mm", "sum"),
    dias_lluvia_fuerte=("lluvia_fuerte", "sum"),
).reset_index()
resumen["lluvia_total_mm"] = resumen["lluvia_total_mm"].round(1)

resumen.sort_values("lluvia_total_mm", ascending=False)

,departamento,lluvia_total_mm,dias_lluvia_fuerte
14,Loreto,1604.6,26
23,Ucayali,1316.0,24
15,Madre de Dios,1248.4,23
24,Áncash,1145.8,9
20,San Martín,1050.5,12
22,Tumbes,931.4,14
10,Junín,850.8,6
7,Huancavelica,817.8,3
4,Cajamarca,796.6,0
17,Pasco,750.4,1


**7. Días sin datos**

In [42]:
print("Días sin dato en total:", lluvia_diaria["lluvia_mm"].isna().sum())

lluvia_diaria.groupby("departamento")["lluvia_mm"].apply(lambda s: s.isna().sum())

Días sin dato en total: 0


departamento
Amazonas         0
Apurímac         0
Arequipa         0
Ayacucho         0
Cajamarca        0
Callao           0
Cusco            0
Huancavelica     0
Huánuco          0
Ica              0
Junín            0
La Libertad      0
Lambayeque       0
Lima             0
Loreto           0
Madre de Dios    0
Moquegua         0
Pasco            0
Piura            0
Puno             0
San Martín       0
Tacna            0
Tumbes           0
Ucayali          0
Áncash           0
Name: lluvia_mm, dtype: int64

Cada capital tiene 151 días (del 1 de enero al 31 de mayo de 2023) y **ningún día llegó sin dato**.

Si hubiera días sin dato, pandas los guarda como `NaN`. `sum` los ignora y la comparación `>= 20` los cuenta como `False`, así que no inflarían ni la lluvia total ni los días de lluvia fuerte.

**8. Guardamos el archivo**

In [43]:
final = capitales.merge(resumen, on="departamento", how="left")
columnas = ["departamento", "capital", "latitud", "longitud",
            "lluvia_total_mm", "dias_lluvia_fuerte"]
final[columnas].to_csv("datos/lluvias_por_departamento.csv", index=False, encoding="utf-8-sig")

pd.read_csv("datos/lluvias_por_departamento.csv")

,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3
1,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9
2,Apurímac,Abancay,-13.63426,-72.88380,726.2,3
3,Arequipa,Arequipa,-16.39899,-71.53747,362.0,1
4,Ayacucho,Ayacucho,-13.16380,-74.22345,498.4,2
5,Cajamarca,Cajamarca,-7.16378,-78.50027,796.6,0
6,Cusco,Cusco,-13.53188,-71.96701,714.0,1
7,Huancavelica,Huancavelica,-12.78691,-74.97298,817.8,3
8,Huánuco,Huánuco,-9.92882,-76.23989,524.3,2
9,Ica,Ica,-14.07538,-75.73422,113.6,0
